In [ ]:
!pip install transformers torch scikit-learn pandas --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import pickle
import os

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Inzira Project/full_dataset.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Keep only opportunity pages for multi-class training
df = df[df['bert_label'] == 1].reset_index(drop=True)

print(f"✓ Dataset loaded: {len(df)} opportunity pages")
print(f"\nCategory distribution:")
print(df['roberta_label'].value_counts().to_string())

✓ Dataset loaded: 393 opportunity pages

Category distribution:
roberta_label
training       61
scholarship    59
job            57
program        57
free_course    54
competition    53
internship     52


In [ ]:
df['text'] = df['text'].fillna('').astype(str)
df['text'] = df['text'].apply(lambda x: ' '.join(x.split()[:400]))

# Encode labels to numbers
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['roberta_label'])

texts  = df['text'].tolist()
labels = df['label_encoded'].tolist()

print(f"✓ Text prepared")
print(f"\nLabel mapping:")
for i, name in enumerate(le.classes_):
    print(f"  {i} = {name}")

# Save label encoder
os.makedirs("/content/drive/MyDrive/Inzira Project/models/roberta_classifier", exist_ok=True)
with open("/content/drive/MyDrive/Inzira Project/models/roberta_classifier/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("\n✓ Label encoder saved")

✓ Text prepared

Label mapping:
  0 = competition
  1 = free_course
  2 = internship
  3 = job
  4 = program
  5 = scholarship
  6 = training

✓ Label encoder saved


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"✓ Dataset split:")
print(f"  Training set   : {len(X_train)} pages")
print(f"  Validation set : {len(X_val)} pages")
print(f"  Test set       : {len(X_test)} pages")

# Create folder first then save
os.makedirs("/content/drive/MyDrive/Inzira Project/checkpoints", exist_ok=True)

with open("/content/drive/MyDrive/Inzira Project/checkpoints/roberta_splits.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train,
        "X_val":   X_val,
        "X_test":  X_test,
        "y_train": y_train,
        "y_val":   y_val,
        "y_test":  y_test
    }, f)
print("✓ Splits saved to Google Drive")

✓ Dataset split:
  Training set   : 275 pages
  Validation set : 59 pages
  Test set       : 59 pages
✓ Splits saved to Google Drive


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
print("✓ RoBERTa tokenizer loaded")

class OpportunityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = OpportunityDataset(X_train, y_train, tokenizer)
val_dataset   = OpportunityDataset(X_val,   y_val,   tokenizer)
test_dataset  = OpportunityDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print("✓ Data loaders ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✓ RoBERTa tokenizer loaded
✓ Data loaders ready


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

NUM_CLASSES = len(le.classes_)
print(f"✓ Number of classes: {NUM_CLASSES}")

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=NUM_CLASSES
)
model = model.to(device)
print("✓ RoBERTa model loaded")

✓ Using device: cuda
✓ Number of classes: 7


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ RoBERTa model loaded


In [ ]:
EPOCHS        = 5
LEARNING_RATE = 2e-5

optimizer   = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"✓ Optimizer ready — training for {EPOCHS} epochs")

✓ Optimizer ready — training for 5 epochs


In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    all_preds  = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch   = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels_batch
            )
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels


print("── STARTING ROBERTA TRAINING ───────────────────────────")
best_val_accuracy = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['label'].to(device)

        model.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if batch_idx % 10 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, device)

    print(f"\n  Epoch {epoch+1} summary:")
    print(f"  Training loss   : {avg_train_loss:.4f}")
    print(f"  Validation loss : {val_loss:.4f}")
    print(f"  Validation acc  : {val_accuracy*100:.2f}%")

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save_pretrained("/content/drive/MyDrive/Inzira Project/models/roberta_classifier")
        tokenizer.save_pretrained("/content/drive/MyDrive/Inzira Project/models/roberta_classifier")
        print(f"  ✓ Best model saved")

── STARTING ROBERTA TRAINING ───────────────────────────
  Epoch 1/5 | Batch 0/18 | Loss: 1.9392
  Epoch 1/5 | Batch 10/18 | Loss: 1.9736

  Epoch 1 summary:
  Training loss   : 1.9567
  Validation loss : 1.9382
  Validation acc  : 18.64%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 2/5 | Batch 0/18 | Loss: 1.9113
  Epoch 2/5 | Batch 10/18 | Loss: 1.8986

  Epoch 2 summary:
  Training loss   : 1.9047
  Validation loss : 1.8426
  Validation acc  : 40.68%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 3/5 | Batch 0/18 | Loss: 1.8511
  Epoch 3/5 | Batch 10/18 | Loss: 1.6528

  Epoch 3 summary:
  Training loss   : 1.6749
  Validation loss : 1.4844
  Validation acc  : 66.10%


...

  ✓ Best model saved
  Epoch 4/5 | Batch 0/18 | Loss: 1.4592
  Epoch 4/5 | Batch 10/18 | Loss: 1.3029

  Epoch 4 summary:
  Training loss   : 1.2928
  Validation loss : 1.1832
  Validation acc  : 71.19%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 5/5 | Batch 0/18 | Loss: 1.2947
  Epoch 5/5 | Batch 10/18 | Loss: 1.0696

  Epoch 5 summary:
  Training loss   : 1.0779
  Validation loss : 1.1001
  Validation acc  : 76.27%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved


In [ ]:
# Load the saved best model and continue training
model = RobertaForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/Inzira Project/models/roberta_classifier"
)
model = model.to(device)
print("✓ Best model loaded")

# Continue training for 3 more epochs
EXTRA_EPOCHS  = 3
optimizer     = AdamW(model.parameters(), lr=1e-5, eps=1e-8)
total_steps   = len(train_loader) * EXTRA_EPOCHS
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("── CONTINUING TRAINING ─────────────────────────────────")
best_val_accuracy = 0

for epoch in range(EXTRA_EPOCHS):
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['label'].to(device)

        model.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if batch_idx % 10 == 0:
            print(f"  Epoch {epoch+1}/{EXTRA_EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, device)

    print(f"\n  Epoch {epoch+1} summary:")
    print(f"  Training loss   : {avg_train_loss:.4f}")
    print(f"  Validation loss : {val_loss:.4f}")
    print(f"  Validation acc  : {val_accuracy*100:.2f}%")

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save_pretrained("/content/drive/MyDrive/Inzira Project/models/roberta_classifier")
        tokenizer.save_pretrained("/content/drive/MyDrive/Inzira Project/models/roberta_classifier")
        print(f"  ✓ Best model saved")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Best model loaded
── CONTINUING TRAINING ─────────────────────────────────
  Epoch 1/3 | Batch 0/18 | Loss: 0.8723
  Epoch 1/3 | Batch 10/18 | Loss: 0.6214

  Epoch 1 summary:
  Training loss   : 0.9772
  Validation loss : 0.9496
  Validation acc  : 74.58%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 2/3 | Batch 0/18 | Loss: 1.1399
  Epoch 2/3 | Batch 10/18 | Loss: 0.8536

  Epoch 2 summary:
  Training loss   : 0.8289
  Validation loss : 0.8629
  Validation acc  : 77.97%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 3/3 | Batch 0/18 | Loss: 0.7465
  Epoch 3/3 | Batch 10/18 | Loss: 0.5265

  Epoch 3 summary:
  Training loss   : 0.6764
  Validation loss : 0.8393
  Validation acc  : 77.97%


In [ ]:
print("── FINAL TEST SET RESULTS ──────────────────────────────")
test_loss, test_accuracy, test_preds, test_labels = evaluate(model, test_loader, device)

f1_macro = f1_score(test_labels, test_preds, average='macro')
f1_weighted = f1_score(test_labels, test_preds, average='weighted')

print(f"\n  Test Accuracy   : {test_accuracy*100:.2f}%")
print(f"  Macro F1 Score  : {f1_macro:.4f}")
print(f"  Weighted F1     : {f1_weighted:.4f}")
print(f"\n  Target: Macro F1 ≥ 0.78")

if f1_macro >= 0.78:
    print("  ✓ TARGET MET — RoBERTa classifier is ready")
else:
    print("  ✗ Target not met")

print("\n── FULL CLASSIFICATION REPORT ──────────────────────────")
print(classification_report(
    test_labels, test_preds,
    target_names=le.classes_
))

── FINAL TEST SET RESULTS ──────────────────────────────

  Test Accuracy   : 84.75%
  Macro F1 Score  : 0.8393
  Weighted F1     : 0.8392

  Target: Macro F1 ≥ 0.78
  ✓ TARGET MET — RoBERTa classifier is ready

── FULL CLASSIFICATION REPORT ──────────────────────────
              precision    recall  f1-score   support

 competition       0.89      1.00      0.94         8
 free_course       0.88      0.88      0.88         8
  internship       1.00      0.88      0.93         8
         job       0.90      1.00      0.95         9
     program       0.80      0.50      0.62         8
 scholarship       0.75      1.00      0.86         9
    training       0.75      0.67      0.71         9

    accuracy                           0.85        59
   macro avg       0.85      0.85      0.84        59
weighted avg       0.85      0.85      0.84        59



In [ ]:
model.save_pretrained("/content/drive/MyDrive/Inzira Project/models/roberta_classifier")
tokenizer.save_pretrained("/content/drive/MyDrive/Inzira Project/models/roberta_classifier")
print("✓ RoBERTa model saved to Google Drive")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ RoBERTa model saved to Google Drive
